# Assignment 6: Diffusion Models

This notebook covers **Denoising Diffusion Probabilistic Models (DDPMs)** — a class of generative models that learn to reverse a gradual noising process.

Topics covered (Lecture 6):
- Linear $\beta$ noise schedule and cumulative $\bar{\alpha}_t$ products
- Closed-form forward process $q(\mathbf{z}_t \mid \mathbf{x}_0)$
- True reverse posterior mean $\tilde{\mu}_t$
- Score / noise prediction intuition
- MLP denoiser with sinusoidal time embeddings (PyTorch)
- DDPM training (Algorithm 1) and sampling (Algorithm 2)
- Connection between diffusion and hierarchical VAEs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (9, 6)

---
## Part 1: Noise Schedule

The **noise schedule** $\{\beta_t\}_{t=1}^T$ controls how much Gaussian noise is injected at each diffusion step.

- $\beta_t \in (0, 1)$: variance of noise added at step $t$
- $\alpha_t = 1 - \beta_t$: fraction of signal kept at step $t$
- $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$: cumulative product — tells us how much signal remains after $t$ steps

A **linear schedule** interpolates $\beta_t$ uniformly from $\beta_{\min}$ to $\beta_{\max}$. At $t=0$, almost no noise is added; at $t=T$, $\bar{\alpha}_T \approx 0$ (fully destroyed signal).

In [ ]:
def linear_beta_schedule(T, beta_start=1e-4, beta_end=0.02):
    """Linear noise schedule from beta_start to beta_end.

    Args:
        T          : int, number of diffusion steps
        beta_start : float, starting beta value
        beta_end   : float, ending beta value

    Returns:
        betas : np.ndarray (T,)
    """
    # TODO: return np.linspace(beta_start, beta_end, T)
    return np.linspace(beta_start, beta_end, T)


def compute_alphas(betas):
    """Compute alpha and alpha_bar sequences from beta schedule.

    Args:
        betas : np.ndarray (T,)

    Returns:
        alphas     : np.ndarray (T,), 1 - betas
        alphas_bar : np.ndarray (T,), cumulative product of alphas
    """
    # TODO:
    # 1. alphas = 1 - betas
    # 2. alphas_bar = np.cumprod(alphas)
    # 3. Return both
    alphas = 1 - betas
    alphas_bar = np.cumprod(alphas)
    return alphas, alphas_bar

In [ ]:
# Sanity check: betas[0] ≈ 1e-4, alphas_bar[-1] ≈ 0
T = 200
betas = linear_beta_schedule(T)
alphas, alphas_bar = compute_alphas(betas)

print(f'betas[0]       = {betas[0]:.6f}  (expected ≈ 1e-4)')
print(f'betas[-1]      = {betas[-1]:.4f}   (expected = 0.02)')
print(f'alphas_bar[0]  = {alphas_bar[0]:.6f}  (expected ≈ 1)')
print(f'alphas_bar[-1] = {alphas_bar[-1]:.6f}  (expected ≈ 0)')

In [ ]:
# Plot the noise schedule
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(betas, color='tab:blue')
axes[0].set_xlabel('Diffusion step $t$')
axes[0].set_ylabel(r'$\beta_t$')
axes[0].set_title('Linear noise schedule $\\beta_t$')
axes[0].grid(True)

axes[1].plot(alphas_bar, color='tab:orange')
axes[1].set_xlabel('Diffusion step $t$')
axes[1].set_ylabel(r'$\bar{\alpha}_t$')
axes[1].set_title(r'Cumulative signal $\bar{\alpha}_t = \prod_{s=1}^t (1 - \beta_s)$')
axes[1].grid(True)

plt.tight_layout()
plt.show()

---
## Part 2: Forward Diffusion Process

The **forward process** gradually corrupts data by adding Gaussian noise.

The single-step transition is:
$$q(\mathbf{z}_t \mid \mathbf{z}_{t-1}) = \mathcal{N}\!\left(\sqrt{1-\beta_t}\,\mathbf{z}_{t-1},\; \beta_t \mathbf{I}\right)$$

Using the reparametrisation trick and the product structure of Gaussians, we can marginalise all intermediate steps to get a **closed-form** for $q(\mathbf{z}_t \mid \mathbf{x}_0)$:

$$q(\mathbf{z}_t \mid \mathbf{x}_0) = \mathcal{N}\!\left(\sqrt{\bar{\alpha}_t}\,\mathbf{x}_0,\; (1-\bar{\alpha}_t)\mathbf{I}\right)$$

This means we can jump directly to any noise level $t$ without iterating:

In [ ]:
def q_sample(x0, t, alphas_bar, eps=None):
    """Sample from q(z_t | x_0) = N(sqrt(alphas_bar[t]) * x0, (1 - alphas_bar[t]) * I).

    Args:
        x0         : np.ndarray (D,) or (N, D)
        t          : int, timestep (0-indexed)
        alphas_bar : np.ndarray (T,)
        eps        : np.ndarray same shape as x0, or None (sample fresh)

    Returns:
        zt  : np.ndarray, noisy sample at step t
        eps : np.ndarray, the noise used
    """
    # TODO:
    # 1. If eps is None: eps = np.random.randn(*x0.shape)
    # 2. sqrt_alpha_bar = np.sqrt(alphas_bar[t])
    # 3. sqrt_one_minus = np.sqrt(1 - alphas_bar[t])
    # 4. zt = sqrt_alpha_bar * x0 + sqrt_one_minus * eps
    # 5. Return zt, eps
    if eps is None:
        eps = np.random.randn(*x0.shape)
    sqrt_alpha_bar = np.sqrt(alphas_bar[t])
    sqrt_one_minus = np.sqrt(1 - alphas_bar[t])
    zt = sqrt_alpha_bar * x0 + sqrt_one_minus * eps
    return zt, eps

In [ ]:
# Sanity check: at t=0, z_t should be close to x0; at t=T-1 it should be near-pure noise
x0 = np.array([3.0, -1.0])

zt_early, _ = q_sample(x0, t=0, alphas_bar=alphas_bar)
zt_late,  _ = q_sample(x0, t=T-1, alphas_bar=alphas_bar)

print(f'x0             = {x0}')
print(f'z_t (t=0)      = {zt_early}  (should be close to x0)')
print(f'z_t (t=T-1)    = {zt_late}   (should look like noise, |values| ~ 1)')

In [ ]:
# Visualise noising trajectory for a single 2D data point
np.random.seed(0)
x0_vis = np.array([3.0, -1.0])
timesteps_to_show = [0, 5, 20, 50, 100, 199]

fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(14, 2.5))
for ax, t_vis in zip(axes, timesteps_to_show):
    zt_vis, _ = q_sample(x0_vis, t=t_vis, alphas_bar=alphas_bar)
    ax.scatter(*x0_vis, color='tab:blue', s=100, label='$x_0$', zorder=3)
    ax.scatter(*zt_vis, color='tab:red',  s=100, label=f'$z_{{{t_vis}}}$', zorder=3)
    ax.set_xlim(-3.5, 5)
    ax.set_ylim(-3.5, 3)
    ax.set_title(f'$t={t_vis}$\n$\\bar{{\\alpha}}_t={alphas_bar[t_vis]:.3f}$', fontsize=9)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, loc='upper right')

plt.suptitle('Closed-form forward process: $x_0$ (blue) → $z_t$ (red)', y=1.02)
plt.tight_layout()
plt.show()

---
## Part 3: Iterative vs. Closed-Form Forward Process

The closed-form $q(\mathbf{z}_t \mid \mathbf{x}_0)$ is derived by composing $t$ Gaussian transitions.  
Concretely, at each step:
$$\mathbf{z}_t = \sqrt{1-\beta_t}\,\mathbf{z}_{t-1} + \sqrt{\beta_t}\,\boldsymbol{\varepsilon}_t, \quad \boldsymbol{\varepsilon}_t \sim \mathcal{N}(0, \mathbf{I})$$

Substituting recursively and using the Gaussian sum rule shows that the marginal is:
$$\mathbf{z}_t = \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0 + \sqrt{1-\bar{\alpha}_t}\,\boldsymbol{\varepsilon}, \quad \boldsymbol{\varepsilon} \sim \mathcal{N}(0, \mathbf{I})$$

Below we verify this empirically: running the iterative and closed-form processes on many samples should give the same distribution (matching mean and variance).

In [ ]:
def q_sample_iterative(x0, T_target, betas):
    """Sample z_{T_target} by iterating q(z_t | z_{t-1}) step by step.

    Args:
        x0       : np.ndarray (D,)
        T_target : int, number of steps to take (0-indexed: steps 0 … T_target-1)
        betas    : np.ndarray (T,)

    Returns:
        zt : np.ndarray (D,)
    """
    # TODO:
    # 1. z = x0.copy()
    # 2. For t in range(T_target):
    #    z = sqrt(1 - betas[t]) * z + sqrt(betas[t]) * np.random.randn(*z.shape)
    # 3. Return z
    z = x0.copy()
    for t in range(T_target):
        z = np.sqrt(1 - betas[t]) * z + np.sqrt(betas[t]) * np.random.randn(*z.shape)
    return z

In [ ]:
# Verify: run both methods many times and check that the resulting distributions match
np.random.seed(7)
x0_check = np.array([2.0, 0.5])
T_target = 100
N_trials = 2000

samples_iterative = np.array([q_sample_iterative(x0_check, T_target, betas) for _ in range(N_trials)])
samples_closed    = np.array([q_sample(x0_check, T_target - 1, alphas_bar)[0] for _ in range(N_trials)])

# Expected mean: sqrt(alphas_bar[T_target-1]) * x0_check
# Expected var:  (1 - alphas_bar[T_target-1]) per dimension
expected_mean = np.sqrt(alphas_bar[T_target - 1]) * x0_check
expected_var  = 1 - alphas_bar[T_target - 1]

print(f'Expected mean:          {expected_mean}')
print(f'Iterative sample mean:  {samples_iterative.mean(axis=0).round(3)}')
print(f'Closed-form sample mean:{samples_closed.mean(axis=0).round(3)}')
print()
print(f'Expected variance (per dim): {expected_var:.4f}')
print(f'Iterative sample var:        {samples_iterative.var(axis=0).round(4)}')
print(f'Closed-form sample var:      {samples_closed.var(axis=0).round(4)}')

---
## Part 4: Reverse Process Posterior Mean

Given $\mathbf{x}_0$ and $\mathbf{z}_t$, the **true reverse posterior** can be computed in closed form using Bayes' rule:

$$q(\mathbf{z}_{t-1} \mid \mathbf{z}_t, \mathbf{x}_0) = \mathcal{N}\!\left(\tilde{\mu}_t(\mathbf{z}_t, \mathbf{x}_0),\; \tilde{\beta}_t \mathbf{I}\right)$$

where:
$$\tilde{\mu}_t = \frac{\sqrt{1-\beta_t}(1 - \bar{\alpha}_{t-1})}{1-\bar{\alpha}_t}\,\mathbf{z}_t + \frac{\sqrt{\bar{\alpha}_{t-1}}\,\beta_t}{1-\bar{\alpha}_t}\,\mathbf{x}_0, \qquad \tilde{\beta}_t = \frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t}\,\beta_t$$

This is the target distribution that the learned reverse model $p_\theta(\mathbf{z}_{t-1} \mid \mathbf{z}_t)$ tries to match at every step.

In [ ]:
def q_posterior_mean(x0, zt, t, betas, alphas_bar):
    """Compute the true reverse posterior mean mu_tilde_t(z_t, x_0).

    Args:
        x0         : np.ndarray (D,)
        zt         : np.ndarray (D,)
        t          : int, current timestep (1-indexed here, so t >= 1)
        betas      : np.ndarray (T,)
        alphas_bar : np.ndarray (T,)

    Returns:
        mu_tilde : np.ndarray (D,)
    """
    # TODO:
    # alpha_bar_t    = alphas_bar[t]
    # alpha_bar_prev = alphas_bar[t-1] if t > 0 else 1.0
    # beta_t         = betas[t]
    # coef_zt = sqrt(1 - beta_t) * (1 - alpha_bar_prev) / (1 - alpha_bar_t)
    # coef_x0 = sqrt(alpha_bar_prev) * beta_t / (1 - alpha_bar_t)
    # return coef_zt * zt + coef_x0 * x0
    alpha_bar_t    = alphas_bar[t]
    alpha_bar_prev = alphas_bar[t - 1] if t > 0 else 1.0
    beta_t         = betas[t]
    coef_zt = np.sqrt(1 - beta_t) * (1 - alpha_bar_prev) / (1 - alpha_bar_t)
    coef_x0 = np.sqrt(alpha_bar_prev) * beta_t / (1 - alpha_bar_t)
    return coef_zt * zt + coef_x0 * x0

In [ ]:
# Sanity check: at t=1 (very early), mu_tilde should be close to x0 (little noise)
# At large t, mu_tilde should be more influenced by z_t
x0_s = np.array([3.0, -1.0])

for t_check in [1, 50, 100, 150, 199]:
    zt_s, _ = q_sample(x0_s, t_check, alphas_bar)
    mu_t = q_posterior_mean(x0_s, zt_s, t_check, betas, alphas_bar)
    print(f't={t_check:3d}  x0={x0_s}  zt={zt_s.round(2)}  mu_tilde={mu_t.round(3)}')

---
## Part 5: Score Intuition — Predicting Noise

The key insight of DDPM is that the denoising network $\varepsilon_\theta$ learns to **predict the noise** $\boldsymbol{\varepsilon}$ that was added to $\mathbf{x}_0$ to produce $\mathbf{z}_t$.

Given a noise prediction $\hat{\boldsymbol{\varepsilon}}$, we can recover a predicted $\hat{\mathbf{x}}_0$:

$$\hat{\mathbf{x}}_0 = \frac{\mathbf{z}_t - \sqrt{1 - \bar{\alpha}_t}\,\hat{\boldsymbol{\varepsilon}}}{\sqrt{\bar{\alpha}_t}}$$

This links to **score matching**: the score $\nabla_{\mathbf{z}_t} \log q(\mathbf{z}_t \mid \mathbf{x}_0) = -\boldsymbol{\varepsilon} / \sqrt{1 - \bar{\alpha}_t}$, so predicting $\boldsymbol{\varepsilon}$ is equivalent to estimating the score.

In [ ]:
def predict_x0_from_noise(xt, t, eps_hat, alphas_bar):
    """Recover predicted x_0 from noise prediction.

    Args:
        xt         : np.ndarray, noisy sample at step t
        t          : int, timestep index
        eps_hat    : np.ndarray, predicted noise (same shape as xt)
        alphas_bar : np.ndarray (T,)

    Returns:
        x0_hat : np.ndarray, predicted clean sample
    """
    # TODO:
    # x0_hat = (xt - sqrt(1 - alphas_bar[t]) * eps_hat) / sqrt(alphas_bar[t])
    x0_hat = (xt - np.sqrt(1 - alphas_bar[t]) * eps_hat) / np.sqrt(alphas_bar[t])
    return x0_hat

In [ ]:
# Sanity check: if we pass in the true noise eps, we should recover x0 exactly
np.random.seed(1)
x0_p5 = np.array([1.5, -2.0])
t_p5  = 80
zt_p5, eps_true = q_sample(x0_p5, t_p5, alphas_bar)

x0_recovered = predict_x0_from_noise(zt_p5, t_p5, eps_true, alphas_bar)
print(f'True x0:        {x0_p5}')
print(f'Recovered x0:   {x0_recovered.round(6)}')
print(f'Max abs error:  {np.abs(x0_p5 - x0_recovered).max():.2e}  (should be ~0)')

---
## Part 6: Denoising Network (PyTorch)

We now build a small **MLP denoiser** $\varepsilon_\theta(\mathbf{z}_t, t)$ that takes a noisy sample and a timestep and predicts the noise.

**Time embedding**: timestep $t$ is encoded via sinusoidal features (similar to positional encodings in Transformers):
$$\text{emb}(t) = \left[\sin\!\left(\frac{t}{T} \cdot 2\pi k\right),\; \cos\!\left(\frac{t}{T} \cdot 2\pi k\right)\right]_{k=1}^{d/2}$$

**Architecture**: $[\mathbf{z}_t \;\|\; \text{emb}(t)] \to \text{Linear} \to \text{ReLU} \to \text{Linear} \to \text{ReLU} \to \text{Linear} \to \hat{\boldsymbol{\varepsilon}}$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
class SimpleDenoiser(nn.Module):
    """Simple MLP denoiser: takes (x_t, t) and predicts noise epsilon.

    Architecture:
        Concatenate x_t (D-dim) and t_emb (emb_dim-dim) -> Linear(D+emb_dim, H) -> ReLU
        -> Linear(H, H) -> ReLU -> Linear(H, D)

    Time is embedded via sinusoidal features:
        t_emb = [sin(t/T * 2pi * k) for k in 1..emb_dim//2] ++ [cos(...)]
    """

    def __init__(self, data_dim, hidden_dim=128, emb_dim=32, T=200):
        super().__init__()
        # TODO: define self.net = nn.Sequential(...)
        # self.T = T; self.emb_dim = emb_dim
        self.T = T
        self.emb_dim = emb_dim
        self.net = nn.Sequential(
            nn.Linear(data_dim + emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, data_dim),
        )

    def time_embedding(self, t_batch):
        """Sinusoidal time embedding.

        Args:
            t_batch : Tensor (B,), integer timesteps

        Returns:
            emb : Tensor (B, emb_dim)
        """
        # TODO:
        # half = self.emb_dim // 2
        # freqs = torch.arange(half, device=t_batch.device) / half
        # angles = t_batch.unsqueeze(1).float() / self.T * 2 * np.pi * (freqs + 1)
        # return torch.cat([torch.sin(angles), torch.cos(angles)], dim=1)
        raise NotImplementedError

    def forward(self, xt, t):
        """
        Args:
            xt : Tensor (B, D)
            t  : Tensor (B,), integer timesteps

        Returns:
            eps_hat : Tensor (B, D)
        """
        # TODO:
        # 1. emb = self.time_embedding(t)
        # 2. x_in = torch.cat([xt, emb], dim=1)
        # 3. return self.net(x_in)
        raise NotImplementedError

### 6.1 Implement `time_embedding` and `forward`

Fill in the two `raise NotImplementedError` stubs above, then run the sanity check below.

In [ ]:
# Sanity check: model should accept (B, D) input and return (B, D) output
# Once you implement time_embedding and forward, this cell should print output shape (4, 2)
torch.manual_seed(0)
_model_test = SimpleDenoiser(data_dim=2, hidden_dim=64, emb_dim=16, T=200).to(device)
_xt_test = torch.randn(4, 2).to(device)
_t_test  = torch.randint(0, 200, (4,)).to(device)

try:
    _out = _model_test(_xt_test, _t_test)
    print(f'Output shape: {_out.shape}  (expected: torch.Size([4, 2]))')
    assert _out.shape == (4, 2), 'Shape mismatch!'
    print('Forward pass OK.')
except NotImplementedError:
    print('NotImplementedError: implement time_embedding and forward first.')

---
## Part 7: DDPM Training Loop

**Algorithm 1 — DDPM Training**:
1. Sample $\mathbf{x}_0 \sim p_{\text{data}}$
2. Sample $t \sim \text{Uniform}\{0, \ldots, T-1\}$
3. Sample $\boldsymbol{\varepsilon} \sim \mathcal{N}(0, \mathbf{I})$
4. Compute $\mathbf{z}_t = \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0 + \sqrt{1-\bar{\alpha}_t}\,\boldsymbol{\varepsilon}$
5. Minimise $\|\boldsymbol{\varepsilon} - \varepsilon_\theta(\mathbf{z}_t, t)\|^2$

We train on a 2D Gaussian mixture (3 components) so we can easily visualise the results.

In [ ]:
# Generate 2D Gaussian mixture dataset (3 components)
np.random.seed(42)

pis_data   = np.array([0.3, 0.4, 0.3])
mus_data   = [np.array([-3.0, -2.0]), np.array([2.0, 2.0]), np.array([0.0, -3.0])]
Sigmas_data = [
    np.array([[1.0, 0.5], [0.5, 0.5]]),
    np.array([[0.5, -0.2], [-0.2, 1.0]]),
    np.array([[1.5, 0.0], [0.0, 0.3]]),
]

N_data = 2000
z_idx = np.random.choice(3, size=N_data, p=pis_data)
X_data = np.vstack([
    np.random.multivariate_normal(mus_data[k], Sigmas_data[k])
    for k in z_idx
]).astype(np.float32)

colors_data = ['tab:blue', 'tab:orange', 'tab:green']
plt.figure()
for k in range(3):
    mask = z_idx == k
    plt.scatter(X_data[mask, 0], X_data[mask, 1], s=10, alpha=0.5,
                color=colors_data[k], label=f'Component {k+1}')
plt.title('Training data: 2D Gaussian mixture')
plt.legend()
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def train_ddpm(X_data, T=200, n_epochs=300, batch_size=256, lr=1e-3, hidden_dim=128):
    """Train a DDPM noise prediction network.

    Algorithm 1:
        repeat:
            x0  ~ data
            t   ~ Uniform({0, ..., T-1})
            eps ~ N(0, I)
            zt  = sqrt(alphas_bar[t]) * x0 + sqrt(1 - alphas_bar[t]) * eps
            loss = ||eps - model(zt, t)||^2
            gradient step

    Args:
        X_data     : np.ndarray (N, D), training data
        T          : int, number of diffusion steps
        n_epochs   : int
        batch_size : int
        lr         : float, learning rate
        hidden_dim : int, MLP hidden width

    Returns:
        model        : trained SimpleDenoiser
        loss_history : list of float, per-epoch average loss
    """
    # Setup noise schedule
    betas_np = linear_beta_schedule(T)
    _, alphas_bar_np = compute_alphas(betas_np)

    # Convert to torch tensors
    betas_t      = torch.tensor(betas_np,      dtype=torch.float32).to(device)
    alphas_bar_t = torch.tensor(alphas_bar_np, dtype=torch.float32).to(device)
    X_t = torch.tensor(X_data, dtype=torch.float32).to(device)

    D = X_data.shape[1]
    model = SimpleDenoiser(data_dim=D, hidden_dim=hidden_dim, T=T).to(device)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)

    loss_history = []
    dataset = torch.utils.data.TensorDataset(X_t)
    loader  = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(n_epochs):
        epoch_loss = 0.0
        for (x0_batch,) in loader:
            # TODO:
            # 1. Sample t uniformly from {0, ..., T-1} — shape (B,)
            # 2. Sample eps ~ N(0, I)                  — shape (B, D)
            # 3. Compute zt = sqrt(alphas_bar_t[t]) * x0_batch + sqrt(1 - alphas_bar_t[t]) * eps
            #    Hint: alphas_bar_t[t] has shape (B,); unsqueeze to broadcast: alphas_bar_t[t].unsqueeze(1)
            # 4. Predict eps_hat = model(zt, t)
            # 5. loss = F.mse_loss(eps_hat, eps)
            # 6. optimiser.zero_grad(); loss.backward(); optimiser.step()
            raise NotImplementedError

        loss_history.append(epoch_loss / len(loader))
        if (epoch + 1) % 50 == 0:
            print(f'Epoch {epoch+1}/{n_epochs}  loss={loss_history[-1]:.4f}')

    return model, loss_history

In [ ]:
# Train the DDPM — complete Part 6 (SimpleDenoiser) and the training loop above first!
torch.manual_seed(42)
try:
    model_trained, loss_history = train_ddpm(X_data, T=200, n_epochs=300, batch_size=256, lr=1e-3)

    plt.figure()
    plt.plot(loss_history, color='tab:blue')
    plt.xlabel('Epoch')
    plt.ylabel('MSE loss')
    plt.title('DDPM training loss')
    plt.grid(True)
    plt.show()
except NotImplementedError:
    print('NotImplementedError: complete SimpleDenoiser and the training loop TODO first.')

---
## Part 8: DDPM Sampling (Algorithm 2)

**Algorithm 2 — DDPM Sampling**:
1. $\mathbf{x}_T \sim \mathcal{N}(0, \mathbf{I})$
2. For $t = T-1, \ldots, 0$:
   - $\mathbf{z} \sim \mathcal{N}(0, \mathbf{I})$ if $t > 0$, else $\mathbf{z} = 0$
   - $\hat{\boldsymbol{\varepsilon}} = \varepsilon_\theta(\mathbf{x}_t, t)$
   - $\hat{\mathbf{x}}_0 = \text{predict\_x0\_from\_noise}(\mathbf{x}_t, t, \hat{\boldsymbol{\varepsilon}})$
   - $\boldsymbol{\mu} = \tilde{\mu}_t(\hat{\mathbf{x}}_0, \mathbf{x}_t)$
   - $\tilde{\beta}_t = \frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t}\,\beta_t$
   - $\mathbf{x}_{t-1} = \boldsymbol{\mu} + \sqrt{\tilde{\beta}_t}\,\mathbf{z}$

In [ ]:
def ddpm_sample(model, n_samples, D, T, betas, alphas_bar):
    """Sample from the DDPM model using the reverse process (Algorithm 2).

    Args:
        model      : trained SimpleDenoiser
        n_samples  : int
        D          : int, data dimension
        T          : int, number of diffusion steps
        betas      : np.ndarray (T,), noise schedule
        alphas_bar : np.ndarray (T,), cumulative alpha products

    Returns:
        samples : np.ndarray (n_samples, D)
    """
    # TODO: implement Algorithm 2
    # Hint: run the loop t = T-1, ..., 0. At each step:
    #   - Convert x_t to torch, call model(xt_torch, t_torch) to get eps_hat
    #   - Convert eps_hat back to numpy
    #   - Use predict_x0_from_noise and q_posterior_mean (numpy functions from above)
    #   - Compute beta_tilde = (1 - alphas_bar[t-1]) / (1 - alphas_bar[t]) * betas[t]  for t > 0
    #   - Add noise: x_{t-1} = mu + sqrt(beta_tilde) * z   (z=0 at t=0)
    raise NotImplementedError

In [ ]:
# Sample 500 points from the trained model and compare to original data
try:
    np.random.seed(0)
    samples = ddpm_sample(model_trained, n_samples=500, D=2, T=200,
                          betas=betas, alphas_bar=alphas_bar)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].scatter(X_data[:, 0], X_data[:, 1], s=8, alpha=0.4, color='tab:blue')
    axes[0].set_title('Training data')
    axes[0].set_aspect('equal')
    axes[0].grid(True, alpha=0.3)

    axes[1].scatter(samples[:, 0], samples[:, 1], s=8, alpha=0.6, color='tab:orange')
    axes[1].set_title('DDPM samples')
    axes[1].set_aspect('equal')
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('Training data vs. DDPM samples')
    plt.tight_layout()
    plt.show()
except NotImplementedError:
    print('NotImplementedError: complete ddpm_sample first.')
except NameError:
    print('NameError: train the model first (run the training cell).')

In [ ]:
# Visualise the denoising trajectory for a single sample
# Show x_T, x_{T//2}, x_{T//4}, x_0
try:
    np.random.seed(5)
    traj_steps = [199, 149, 99, 49, 0]

    def ddpm_sample_trajectory(model, D, T, betas, alphas_bar, record_steps):
        """Like ddpm_sample but records x_t at specified steps for a single sample."""
        model.eval()
        x = np.random.randn(1, D)
        trajectory = {}
        with torch.no_grad():
            for t in reversed(range(T)):
                xt_torch = torch.tensor(x, dtype=torch.float32).to(device)
                t_torch  = torch.tensor([t], dtype=torch.long).to(device)
                eps_hat  = model(xt_torch, t_torch).cpu().numpy()

                x0_hat = predict_x0_from_noise(x, t, eps_hat, alphas_bar)
                if t > 0:
                    mu = q_posterior_mean(x0_hat[0], x[0], t, betas, alphas_bar)[np.newaxis]
                    alpha_bar_prev = alphas_bar[t - 1]
                    beta_tilde = (1 - alpha_bar_prev) / (1 - alphas_bar[t]) * betas[t]
                    x = mu + np.sqrt(beta_tilde) * np.random.randn(1, D)
                else:
                    x = x0_hat

                if t in record_steps:
                    trajectory[t] = x[0].copy()
        return trajectory

    traj = ddpm_sample_trajectory(model_trained, D=2, T=200,
                                  betas=betas, alphas_bar=alphas_bar,
                                  record_steps=set(traj_steps))

    fig, axes = plt.subplots(1, len(traj_steps), figsize=(14, 3))
    for ax, t_show in zip(axes, traj_steps):
        pt = traj[t_show]
        ax.scatter(*pt, s=120, color='tab:red', zorder=5)
        ax.scatter(X_data[:, 0], X_data[:, 1], s=4, alpha=0.15, color='tab:blue')
        ax.set_title(f'$t={t_show}$', fontsize=10)
        ax.set_xlim(-7, 6); ax.set_ylim(-6, 5)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.2)
    plt.suptitle('Denoising trajectory (red dot = single sample, blue = data)', y=1.02)
    plt.tight_layout()
    plt.show()
except NotImplementedError:
    print('NotImplementedError: complete SimpleDenoiser, training loop, and ddpm_sample first.')
except NameError:
    print('NameError: train the model first.')

---
## Part 9: Connecting to Hierarchical VAEs

Diffusion models can be viewed as a **Hierarchical VAE (HVAE)** with $T$ latent layers $\mathbf{z}_1, \ldots, \mathbf{z}_T$:

- The **encoder** $q(\mathbf{z}_{1:T} \mid \mathbf{x}_0)$ is the **fixed** forward noising process — no parameters to learn.
- The **decoder** $p_\theta(\mathbf{x}_0 \mid \mathbf{z}_{1:T})$ is the learned reverse process — $T$ denoising steps.
- The ELBO decomposes into $T+1$ terms: a reconstruction term $L_0$ and KL terms $L_1, \ldots, L_T$.

Because the encoder is fixed (and Gaussian at each level), the KL terms $L_t$ become simple MSE objectives — which is exactly the DDPM training loss $\|\boldsymbol{\varepsilon} - \varepsilon_\theta(\mathbf{z}_t, t)\|^2$.

| Aspect | Standard VAE | Diffusion HVAE |
|---|---|---|
| Encoder $q(\mathbf{z} \mid \mathbf{x})$ | Learned neural net | Fixed Gaussian noise process |
| Decoder $p(\mathbf{x} \mid \mathbf{z})$ | Single layer, learned | $T$ reverse steps, learned |
| Latent $\mathbf{z}$ | Single $\mathbf{z} \sim \mathcal{N}(0,\mathbf{I})$ | Sequence $\mathbf{z}_T, \ldots, \mathbf{z}_1$ |
| ELBO terms | Reconstruction $+$ $\text{KL}[q \| \mathcal{N}(0,\mathbf{I})]$ | $L_0 + \sum_t L_t + L_T$ |
| Sampling | Single decoder forward pass | $T$ denoising steps |

**Reflection question**: Why do we train the network to predict $\boldsymbol{\varepsilon}$ (the noise) rather than directly predicting $\mathbf{x}_0$?

Think about the signal-to-noise ratio at different timesteps, what the network "sees" as input, and what the gradient signal looks like early vs. late in training.

**Your answer**: TODO

---
## Summary

In this notebook you:

- Implemented the **linear $\beta$ noise schedule** and derived the cumulative $\bar{\alpha}_t$ products
- Built the **closed-form forward process** $q(\mathbf{z}_t \mid \mathbf{x}_0)$ and verified it matches the iterative version
- Derived the **true reverse posterior mean** $\tilde{\mu}_t(\mathbf{z}_t, \mathbf{x}_0)$
- Connected noise prediction to the **score function**
- Built a **sinusoidal time-embedding MLP denoiser** in PyTorch
- Trained DDPM on a 2D dataset (**Algorithm 1**) and sampled from it (**Algorithm 2**)
- Connected diffusion to **hierarchical VAEs**

| Component | Role |
|---|---|
| `linear_beta_schedule` | Defines how fast noise is added |
| `compute_alphas` | Derives cumulative signal decay $\bar{\alpha}_t$ |
| `q_sample` | Closed-form jump to any noise level $t$ |
| `q_sample_iterative` | Iterative forward process (verification) |
| `q_posterior_mean` | True reverse posterior mean $\tilde{\mu}_t$ |
| `predict_x0_from_noise` | Noise $\to$ clean data inversion |
| `SimpleDenoiser` | MLP with sinusoidal time embedding |
| `train_ddpm` | Algorithm 1 — learn to predict noise |
| `ddpm_sample` | Algorithm 2 — reverse denoising chain |

**Next**: Assignment 7 dives into the score-based perspective that underlies diffusion models.